# Medical data: exploratory and interaction-information analysis

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import copy
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from dotenv import load_dotenv
from sklearn import set_config

from my_project.data.utils import columns_info_func, load_column_config
from my_project.experiments.medical import (
    medical_discretization_pipeline,
    medical_preprocessing_pipeline,
)
from my_project.information_theory import compute_information_matrices
from my_project.information_theory.estimators import _mi_pair, encode_features
from my_project.reporting import (
    CATEGORICAL_PALETTE,
    CATEGORY_LABELS_PL,
    PLOT_LABELS_PL,
    SEQUENTIAL_PALETTE,
    configure_notebook_plots,
    pairwise_ii_mi_table,
    plot_category_barplots,
    plot_class_balance,
    plot_feature_histograms,
    plot_ii_by_edges,
    plot_ii_graph,
    save_figure,
    save_table,
)

RANDOM_STATE = 42

# Preserve DataFrame outputs so feature names remain aligned with matrices and graphs.
set_config(transform_output="pandas")
pd.options.display.max_columns = None

configure_notebook_plots()
PALETTE = CATEGORICAL_PALETTE
PALETTE_SEQ = SEQUENTIAL_PALETTE

In [ ]:
# Eksport do LaTeX-a — ryciny -> $RESULTS_LATEX_DIR/plots/*.pdf,
# tabele -> $RESULTS_LATEX_DIR/tables/*.tex.
load_dotenv(override=True)

DATA_DIR = os.getenv("DATA_DIR")
CONFIGS_DIR = os.getenv("CONFIGS_DIR")
RESULTS_LATEX_DIR = os.getenv("RESULTS_LATEX_DIR")
RESULTS_DIR = os.getenv("RESULTS_DIR")

with open(os.path.join(CONFIGS_DIR, "data.yaml")) as f:
    data_config = yaml.safe_load(f)

col_cfg = load_column_config(CONFIGS_DIR)

# Compatibility aliases used throughout this exploratory notebook.
POLISH_COLUMN_NAMES = PLOT_LABELS_PL
POLISH_CATEGORY_NAMES = CATEGORY_LABELS_PL

## Data loading and analysis subset

The cleaned dataset and predefined train/validation/test indices are loaded from the configured data directory. Modeling columns come from `configs/columns.py`.

Descriptive and information-theoretic analyses use the training subset only. `X_all` and `y_all` retain the full cohort for the predefined cross-validation folds, where each preprocessing pipeline is fitted on the corresponding training fold.

In [ ]:
df = pd.read_pickle(
    os.path.join(DATA_DIR, data_config["paths"]["preprocessed_data_path"])
)

# Use the same predefined split as the modeling workflow.
split_index_dict = pd.read_pickle(
    os.path.join(DATA_DIR, data_config["paths"]["split_indices_path"])
)
train_idx = split_index_dict["train"]
val_idx = split_index_dict["valid"]

train_val_idx = np.concatenate([train_idx, val_idx])

y = df[col_cfg.TARGET].astype(int)
# X = select_modeling_features(df, CONFIGS_DIR)
X = df.copy().drop(columns=[col_cfg.TARGET])

# Retain the complete cohort for analyses based on predefined CV folds.
X_all, y_all = X, y

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
X.head()

In [ ]:
X["RRRozkurcz"].value_counts()

In [ ]:
SELECTED_FEATURES = col_cfg.MODELING_FEATURES

selected_features = [c for c in SELECTED_FEATURES if c in X.columns]
missing_features = [c for c in SELECTED_FEATURES if c not in X.columns]
if missing_features:
    print(f"Skipping {len(missing_features)} unavailable features: {missing_features}")

X = X.loc[:, selected_features]

# Main EDA uses training rows only; full rows remain available for fold analysis.
X_all, y_all = X, y
X = X_all.loc[train_val_idx]
y = y_all.loc[train_val_idx]

print(f"X (train): {X.shape} | X_all (train+valid+test): {X_all.shape}")

In [ ]:
X.isna().mean().sort_values(ascending=False).head(30)

## Cohort overview and temporal diagnostics

This section summarizes class balance, admission volume, mortality over time, and time-dependent missingness. Temporal missingness is especially important for `tro`, `trot`, and `crp`, because assay availability and clinical practice changed during data collection.

In [ ]:
df["DataPrzyjecia"].min(), df["DataPrzyjecia"].max()

In [ ]:
df["DataPrzyjecia"] = pd.to_datetime(df["DataPrzyjecia"], format="%Y-%m-%d")
df["MiesiacRokPrzyjecia"] = df["DataPrzyjecia"].dt.to_period("M")

In [ ]:
monthly = (
    df.groupby("MiesiacRokPrzyjecia")
    .agg(count=("zgon_binary", "size"), zgon_rate=("zgon_binary", "mean"))
    .sort_index()
)
x = monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x, monthly["count"], color="steelblue", label="Liczność pacjentów")
ax.set_xlabel("Miesiąc przyjęcia")
ax.set_ylabel("Liczność pacjentów")

# Show every third month on the X-axis to prevent label overlap.
tick_step = 3
ax.set_xticks(range(0, len(x), tick_step))
ax.set_xticklabels(x[::tick_step], rotation=90)

ax2 = ax.twinx()
ax2.plot(x, monthly["zgon_rate"], color="crimson", marker="o", label="Odsetek zgonów")
ax2.set_ylabel("Odsetek zgonów")
ax2.grid(
    False
)  # Disable the secondary-axis grid to avoid overlapping the primary grid.

handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax.legend(handles + handles2, labels + labels2, loc="upper left")

fig.tight_layout()
save_figure(fig, "medical_eda_monthly_admissions_mortality")
plt.show()

In [ ]:
df["zgon_binary"].mean()

In [ ]:
df["trot_na"] = df["trot"].isna().astype(int)
df["tro_na"] = df["tro"].isna().astype(int)

df_agg = df.groupby("MiesiacRokPrzyjecia")[["trot_na", "tro_na"]].mean().reset_index()

In [ ]:
plt.plot(df_agg["MiesiacRokPrzyjecia"].astype(str), df_agg["trot_na"], label="trot_na")
plt.plot(df_agg["MiesiacRokPrzyjecia"].astype(str), df_agg["tro_na"], label="tro_na")
plt.xlabel("Miesiąc przyjęcia")
plt.ylabel("Odsetek braków")
plt.title("Odsetek braków w cechach 'trot' i 'tro' w czasie")
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()

In [ ]:
var = "chol"

df[var + "_na"] = df[var].isna().astype(int)
df_agg = df.groupby("MiesiacRokPrzyjecia")[[var + "_na"]].mean().reset_index()

plt.plot(
    df_agg["MiesiacRokPrzyjecia"].astype(str), df_agg[var + "_na"], label=var + "_na"
)
plt.xlabel("Miesiąc przyjęcia")
plt.ylabel("Odsetek braków")
plt.title("Odsetek braków w cechach 'crp' w czasie")
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()

In [ ]:
var = "ldl"

df[var + "_na"] = df[var].isna().astype(int)
df_agg = df.groupby("MiesiacRokPrzyjecia")[[var + "_na"]].mean().reset_index()

plt.plot(
    df_agg["MiesiacRokPrzyjecia"].astype(str), df_agg[var + "_na"], label=var + "_na"
)
plt.xlabel("Miesiąc przyjęcia")
plt.ylabel("Odsetek braków")
plt.title("Odsetek braków w cechach 'crp' w czasie")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()

In [ ]:
var = "crp"

df[var + "_na"] = df[var].isna().astype(int)
df_agg = df.groupby("MiesiacRokPrzyjecia")[[var + "_na"]].mean().reset_index()

plt.plot(
    df_agg["MiesiacRokPrzyjecia"].astype(str), df_agg[var + "_na"], label=var + "_na"
)
plt.xlabel("Miesiąc przyjęcia")
plt.ylabel("Odsetek braków")
plt.title("Odsetek braków w cechach 'crp' w czasie")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()

### Availability of features over time

For each modeling feature, monthly availability is defined as the proportion of non-missing observations. A candidate introduction date is the split with the largest increase in mean availability, requiring at least six months of data before and after the split. Features are flagged as likely introduced later when availability increases by at least 25 percentage points, from at most 50% before the split to at least 50% afterward.

This is a screening analysis rather than proof of a formal protocol change: abrupt shifts may also reflect changes in patient selection, data extraction, or documentation.

In [ ]:
analysis_features = [column for column in X_all.columns if column in df.columns]
monthly_availability = (
    df[analysis_features]
    .notna()
    .assign(month=df["MiesiacRokPrzyjecia"])
    .groupby("month")
    .mean()
    .T
)

min_months_per_period = 6
change_records = []
for feature, availability in monthly_availability.iterrows():
    candidate_splits = range(
        min_months_per_period,
        len(availability) - min_months_per_period + 1,
    )
    gains = [
        availability.iloc[split:].mean() - availability.iloc[:split].mean()
        for split in candidate_splits
    ]
    best_split = list(candidate_splits)[int(np.argmax(gains))]
    observed_months = availability.index[availability.gt(0)]
    change_records.append(
        {
            "feature": feature,
            "first_observed_month": (
                str(observed_months[0]) if len(observed_months) else None
            ),
            "estimated_use_start": str(availability.index[best_split]),
            "availability_before": availability.iloc[:best_split].mean(),
            "availability_after": availability.iloc[best_split:].mean(),
            "availability_gain": gains[best_split - min_months_per_period],
        }
    )

availability_changes = (
    pd.DataFrame(change_records)
    .sort_values("availability_gain", ascending=False)
    .reset_index(drop=True)
)
likely_introduced_later = availability_changes.query(
    "availability_before <= 0.5 and availability_after >= 0.5 and availability_gain >= 0.25"
)
save_table(
    likely_introduced_later.round(3).rename(columns=POLISH_COLUMN_NAMES),
    "medical_eda_features_introduced_over_time",
)

feature_order = availability_changes["feature"]
fig, ax = plt.subplots(figsize=(14, max(8, 0.28 * len(feature_order))))
sns.heatmap(
    monthly_availability.loc[feature_order],
    vmin=0,
    vmax=1,
    cmap="viridis",
    cbar_kws={"label": "Udział obserwacji bez braków"},
    ax=ax,
)
tick_positions = np.arange(0, monthly_availability.shape[1], 3)
ax.set_xticks(tick_positions + 0.5)
ax.set_xticklabels(
    monthly_availability.columns.astype(str)[tick_positions],
    rotation=90,
)
ax.set_xlabel("Miesiąc przyjęcia")
ax.set_ylabel("Cecha")
ax.set_title("Miesięczna dostępność cech")
fig.tight_layout()
save_figure(fig, "medical_eda_feature_availability_over_time")
plt.show()

likely_introduced_later

## Troponin assays and CRP

Two troponin assays were used during data collection:

- **TnI-Ultra (Siemens)** from January 2012 to September 2014: limit of detection `0.006 ng/mL`, 99th-percentile reference limit `0.040 ng/mL`.
- **High-sensitivity cTnT (Roche)** from October 2014 onward: limit of detection `0.005 ng/mL`, 99th-percentile reference limit `0.014 ng/mL`.

The raw columns `tro` and `trot` are therefore inspected separately. The modeling pipeline later discretizes each assay and merges them into `troponin_category`; column names in the source data remain unchanged. CRP is analyzed alongside troponin because its high missingness is retained as an explicit category by the medical pipeline.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(df, x="crp", bins=30, kde=True, color="steelblue", ax=ax)
ax.set_title("Rozkład CRP")
ax.set_xlabel("crp")
ax.set_ylabel("Liczba obserwacji")
fig.tight_layout()
plt.show()

In [ ]:
df.groupby("TroponinaT", dropna=False)[["tro", "trot"]].agg(
    ["mean", "min", "max", "count"]
)

## Descriptive statistics

The following tables summarize distributions, missingness, data types, and class balance for the training subset before preprocessing.

In [ ]:
# Detailed quantiles for the two raw troponin assays.
descr = X.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
descr.loc[["tro", "trot"]]

In [ ]:
# Missing values, data types, and category counts for each feature.
columns_info = columns_info_func(X)
columns_info.head(20)

In [ ]:
# Raw categorical and binary features before imputation or grouping.
X_categorical = X.select_dtypes(
    include=["object", "category", "string", "bool", "boolean"]
)

categorical_stats = pd.DataFrame(
    {
        "n_classes": X_categorical.nunique(dropna=True),
        "missing_pct": X_categorical.isna().mean(),
        "most_frequent": X_categorical.apply(
            lambda column: (
                column.value_counts(normalize=True, dropna=True).max()
                if column.notna().any()
                else np.nan
            )
        ),
        "least_frequent": X_categorical.apply(
            lambda column: (
                column.value_counts(normalize=True, dropna=True).min()
                if column.notna().any()
                else np.nan
            )
        ),
    }
)
categorical_stats.index.name = "feature"
categorical_stats = categorical_stats.sort_index()

save_table(
    categorical_stats.round(3).rename(columns=POLISH_COLUMN_NAMES),
    "medical_eda_categorical_stats",
)
categorical_stats

In [ ]:
# Binary target balance (1 = death).
class_counts = y.value_counts().sort_index()
print(class_counts)
print(f"\nClass 1 share: {y.mean():.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
plot_class_balance(y, ax=ax, xlabel="zgon_binary", ylabel="liczba pacjentów")
fig.tight_layout()
plt.show()

## Raw feature distributions

The plots below use the training subset before imputation, clipping, merging, discretization, or encoding. Numeric columns are shown as histograms and categorical columns as raw category proportions. Missing values remain unchanged in `X` and are omitted from these distribution plots.

In [ ]:
# Raw training features only: no imputation, merging, discretization, or encoding.
continuous_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
boolean_cols = X.select_dtypes(include=["bool", "boolean"]).columns.tolist()
CATEGORY_MIN_FREQUENCY = 0.05

print(f"Continuous raw features: {len(continuous_cols)}")
print(f"Categorical raw features: {len(categorical_cols)}")
print(f"Boolean raw features: {len(boolean_cols)}")

In [ ]:
# Adaptively trim only extreme tails; source values remain unchanged.

HISTOGRAM_FONT_SIZE = 25
HISTOGRAM_TITLE_PAD = 14
HISTOGRAM_SUBPLOT_SIZE = (6, 5)
HISTOGRAM_WSPACE = 0.4
HISTOGRAM_HSPACE = 0.5

fig = plot_feature_histograms(
    X=X,
    y=None,
    columns=sorted(continuous_cols),
    trim_quantiles=(0.01, 0.99),
    tail_iqr_multiplier=3.0,
    n_cols=6,
    bins=20,
    font_size=HISTOGRAM_FONT_SIZE,
    title_pad=HISTOGRAM_TITLE_PAD,
    subplot_size=HISTOGRAM_SUBPLOT_SIZE,
    wspace=HISTOGRAM_WSPACE,
    hspace=HISTOGRAM_HSPACE,
)
save_figure(fig, "medical_eda_raw_feature_histograms")
plt.show()
len(continuous_cols)

In [ ]:
X_numeric = X[sorted(continuous_cols)]

# Descriptive feature statistics: mean, median, missing count, and unique-value count.
descriptive_stats = pd.DataFrame(
    {
        "mean": X_numeric.mean(),
        "median": X_numeric.median(),
        "std": X_numeric.std(),
        "min": X_numeric.min(),
        "max": X_numeric.max(),
        "skewness": X_numeric.skew(),
        "missing_pct": X_numeric.isna().mean(),
        # "n_unique": X_numeric.nunique(),
    }
)
descriptive_stats.index.name = "feature"
descriptive_stats = descriptive_stats.sort_index()
save_table(
    descriptive_stats.round(3).rename(columns=POLISH_COLUMN_NAMES).reset_index(),
    "medical_eda_descriptive_stats",
    index=False,
)
descriptive_stats.round(3).rename(columns=POLISH_COLUMN_NAMES).reset_index()

descriptive_stats

In [ ]:
# Reconstructed data dictionary: interpretations are inferred rather than sourced.
data_dictionary = (
    pd.read_csv(os.path.join(CONFIGS_DIR, "medical_data_dictionary.csv"))
    .set_index("zmienna")
    .fillna("")
)
data_dictionary.index.name = "zmienna"

print("Poziomy pewności:")
print(data_dictionary["pewność"].value_counts().rename("Liczność"))
save_table(
    data_dictionary.rename(columns=POLISH_COLUMN_NAMES),
    "medical_data_dictionary",
)
data_dictionary

In [ ]:
data_dictionary = data_dictionary.loc[X.columns]

In [ ]:
data_dictionary[data_dictionary["pewność"] == "niejednoznaczna"]

In [ ]:
dtypes = X.dtypes.astype(str)

dtypes_dict = {
    "float64": "ciągła",
    "int64": "całkowita",
    "object": "kategoryczna",
    "boolean": "binarna",
}

dtypes.map(dtypes_dict).to_frame().join(dtypes.to_frame("oryginalny_typ"))

dtypes = dtypes.map(dtypes_dict).sort_index()
dtypes.name = "typ zmiennej"

In [ ]:
# Continuous features after median imputation and Yeo-Johnson transformation.
preprocessing_pipeline = copy.deepcopy(medical_preprocessing_pipeline)
X_preprocessed = preprocessing_pipeline.fit_transform(X, y)
transformed_continuous_cols = X_preprocessed.select_dtypes(
    include=[np.number]
).columns.tolist()

print(f"Continuous transformed features: {len(transformed_continuous_cols)}")
fig = plot_feature_histograms(
    X=X_preprocessed,
    y=None,
    columns=transformed_continuous_cols,
    trim_quantiles=(0.005, 0.995),
    tail_iqr_multiplier=3.0,
    bins=20,
    n_cols=7,
)
save_figure(fig, "medical_eda_transformed_feature_histograms")
plt.show()

In [ ]:
df.shape

In [ ]:
# Raw category proportions; levels below 1% are merged into the "INNE" category.
raw_category_cols = categorical_cols + boolean_cols
X_categories_pl = (
    X.loc[:, raw_category_cols].astype(object).replace(POLISH_CATEGORY_NAMES)
)
CATEGORY_PLOT_FONT_SIZE = 15
CATEGORY_PLOT_TITLE_PAD = 12
CATEGORY_PLOT_SUBPLOT_SIZE = (6, 4)
CATEGORY_PLOT_WSPACE = 0.3
CATEGORY_PLOT_HSPACE = 0.65
CATEGORY_PLOT_LABEL_WRAP_WIDTH = 6
fig = plot_category_barplots(
    X_categories_pl,
    y=None,
    columns=raw_category_cols,
    n_cols=np.min([len(raw_category_cols), 3]),
    subplot_size=CATEGORY_PLOT_SUBPLOT_SIZE,
    wspace=CATEGORY_PLOT_WSPACE,
    hspace=CATEGORY_PLOT_HSPACE,
    font_size=CATEGORY_PLOT_FONT_SIZE,
    title_pad=CATEGORY_PLOT_TITLE_PAD,
    min_frequency=CATEGORY_MIN_FREQUENCY,
    other_label="INNE",
    x_ticks_angle=45,
    x_tick_wrap_width=CATEGORY_PLOT_LABEL_WRAP_WIDTH,
)
save_figure(fig, "medical_eda_raw_category_barplots")
plt.show()
len(raw_category_cols)

In [ ]:
preprocessed_category_cols = X_preprocessed.select_dtypes(
    include=["object", "category", "string", "bool", "boolean"]
).columns.tolist()
X_preprocessed_categories_pl = (
    X_preprocessed.loc[:, preprocessed_category_cols]
    .astype(object)
    .replace(POLISH_CATEGORY_NAMES)
)

fig = plot_category_barplots(
    X_preprocessed_categories_pl,
    y=None,
    columns=preprocessed_category_cols,
    n_cols=3,
    # min_frequency=CATEGORY_MIN_FREQUENCY,
    other_label="INNE",
)
save_figure(fig, "medical_eda_transformed_category_barplots")
plt.show()

## Discrete representation for information-theoretic analysis

The medical discretization pipeline is fitted on the analysis subset. It imputes low-missing features, preserves informative missingness, merges `tro` and `trot` into `troponin_category`, and quantile-bins the remaining numerical variables. The resulting frame is used consistently in all information-theoretic calculations.

In [ ]:
disc_pipeline = copy.deepcopy(medical_discretization_pipeline)

X_disc = disc_pipeline.fit_transform(X, y)

X_disc = X_disc.reindex(sorted(X_disc.columns), axis=1)


print(f"X_disc shape: {X_disc.shape}")

X_disc.head()

In [ ]:
X_disc.nunique().sort_values()

## Information matrices

For every feature pair, `compute_information_matrices` calculates:

- feature mutual information, $I(X_i; X_j)$;
- joint target information, $I((X_i, X_j); y)$;
- interaction information, $II(X_i; X_j; y)$.

Under the convention used here, positive interaction information indicates synergy with respect to the target. The input is already discrete, so no additional binning is requested.

In [ ]:
matrices = compute_information_matrices(
    X_disc,
    y,
    measures=("feature_mi", "joint_target_mi", "interaction_information"),
    n_jobs=-1,
)
matrices.keys()

In [ ]:
figsize = (16, 14)

scale = 0.6
figsize = (figsize[0] * scale, figsize[1] * scale)

In [ ]:
# Interaction information matrix II(Xi; Xj; y).
ii = matrices["joint_target_mi"]

fig, ax = plt.subplots(figsize=figsize)
sns.heatmap(ii, cmap="viridis", square=True, ax=ax)
fig.tight_layout()
save_figure(fig, "medical_eda_joint_target_mi")
plt.show()

In [ ]:
# Interaction information matrix II(Xi; Xj; y).
ii = matrices["interaction_information"]
ii_vmin = np.nanmin(ii.to_numpy())
ii_vmax = np.nanmax(ii.to_numpy())

fig, ax = plt.subplots(figsize=figsize)
sns.heatmap(
    ii,
    cmap="viridis",
    vmin=ii_vmin,
    vmax=ii_vmax,
    square=True,
    ax=ax,
)
fig.tight_layout()
save_figure(fig, "medical_eda_interaction_information")
plt.show()

In [ ]:
# Mutual information between features I(Xi; Xj).
fig, ax = plt.subplots(figsize=figsize)
sns.heatmap(matrices["feature_mi"], cmap="viridis", square=True, ax=ax)
fig.tight_layout()
save_figure(fig, "medical_eda_mutual_information")
plt.show()

In [ ]:
# Feature-pair table with II(Xi; Xj; y) and I(Xi; Xj), sorted by decreasing II.
pair_table = pairwise_ii_mi_table(ii, matrices["feature_mi"])
pair_table_out = pair_table.rename(
    columns={
        "feature_i": "zmienna_i",
        "feature_j": "zmienna_j",
        "interaction_information": "informacja_interakcyjna",
    }
).copy()

pair_table_out.index = pair_table_out.index + 1

save_table(
    pair_table_out.head(20).round(4).drop(columns=["mutual_information"]),
    "medical_eda_pairwise_ii_mi",
    # index=False
)


pair_table_out.to_csv(os.path.join(RESULTS_DIR, "pair_table_full.csv"), index=False)

## Feature-pair ranking by interaction information

The plot below presents the 20 feature pairs with the highest $II(X_i; X_j; y)$ values.
Positive values indicate feature-pair synergy with respect to the target variable.

In [ ]:
# Feature pairs with the highest interaction information with respect to the target.
TOP_N_PAIRS = 20
interaction_top = pair_table.head(TOP_N_PAIRS).copy()
interaction_top["pair"] = (
    interaction_top["feature_i"] + " + " + interaction_top["feature_j"]
)
interaction_top = interaction_top.sort_values("interaction_information", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.get_cmap(PALETTE_SEQ)(
    plt.Normalize()(interaction_top["interaction_information"].to_numpy())
)
bars = ax.barh(
    interaction_top["pair"],
    interaction_top["interaction_information"],
    color=colors,
    edgecolor="white",
    linewidth=0.5,
)
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
ax.set_xlabel(r"Informacja interakcyjna $II(X_i; X_j; y)$ [bity]")
ax.set_ylabel("")
ax.set_title(f"Pary cech o najwyższej informacji interakcyjnej (top {TOP_N_PAIRS})")
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.margins(x=0.12)
fig.tight_layout()
save_figure(fig, "medical_eda_top_interaction_pairs")
plt.show()

## Feature rankings

Features are ranked in two complementary ways: individual mutual information with the target, $I(X_i; y)$, and the strength of their interaction-information pairs. Pair-based summaries report each feature's maximum, mean, and total interaction information across all partners.

In [ ]:
# I(Xi; y): mutual information between each feature and the target (using the same
# plug-in estimator in bits as the matrices). Features are sorted by target MI.
X_enc = encode_features(X_disc)
y_codes = pd.factorize(y)[0]

feature_target_mi = (
    pd.Series(
        {col: _mi_pair(X_enc[col].to_numpy(), y_codes) for col in X_enc.columns},
        name="mutual_information",
    )
    .sort_values(ascending=False)
    .rename_axis("feature")
    .reset_index()
)
feature_target_mi

In [ ]:
# Horizontal MI ranking: bar length and color encode association strength with the target.
mi_plot = feature_target_mi.sort_values("mutual_information", ascending=True).copy()
mi_plot["feature_label"] = (
    mi_plot["feature"].map(POLISH_COLUMN_NAMES).fillna(mi_plot["feature"])
)

fig_height = max(5, 0.32 * len(mi_plot))
fig, ax = plt.subplots(figsize=(10, fig_height))
colors = plt.get_cmap(PALETTE_SEQ)(
    plt.Normalize()(mi_plot["mutual_information"].to_numpy())
)
bars = ax.barh(
    mi_plot["feature_label"],
    mi_plot["mutual_information"],
    color=colors,
    edgecolor="white",
    linewidth=0.5,
)
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
ax.set_xlabel("Informacja wzajemna z celem [bity]")
ax.set_ylabel("")
ax.set_title("Informacja wzajemna między cechami a zmienną docelową")
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.margins(x=0.1)
fig.tight_layout()
save_figure(fig, "medical_eda_feature_target_mi")
plt.show()

In [ ]:
# Features ranked by the interaction information of their pairs II(Xi; Xj; y), across ALL pairs.
# For each feature: strongest-pair score (ii_max), mean, and sum across its pairs.
feature_ii = (
    pd.concat(
        [
            pair_table[["feature_i", "interaction_information"]].rename(
                columns={"feature_i": "feature"}
            ),
            pair_table[["feature_j", "interaction_information"]].rename(
                columns={"feature_j": "feature"}
            ),
        ]
    )
    .groupby("feature")["interaction_information"]
    .agg(ii_max="max", ii_mean="mean", ii_sum="sum")
    .sort_values("ii_max", ascending=False)
    .reset_index()
)

# Combined table: target MI and pairwise II aggregates, one row per feature.
feature_ranking = feature_target_mi.merge(feature_ii, on="feature", how="outer")
feature_ranking = feature_ranking.sort_values(
    "mutual_information", ascending=False
).reset_index(drop=True)
save_table(
    feature_ranking.round(3).rename(columns=POLISH_COLUMN_NAMES),
    "medical_eda_feature_ranking",
)
feature_ranking

In [ ]:
# Line plot: all features sorted by the maximum II of their pairs in decreasing order.
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    feature_ii["feature"],
    feature_ii["ii_max"],
    marker="o",
    markersize=3,
    linewidth=1,
)
ax.set_xlabel("cecha")
ax.set_ylabel("maks. II(Xi; Xj; y) po parach")
ax.set_title("Cechy wg najsilniejszej pary (II)")
ax.tick_params(axis="x", rotation=90, labelsize=6)
fig.tight_layout()
save_figure(fig, "medical_eda_feature_ii_pairs_line")
plt.show()

In [ ]:
# Interaction information of feature-pair edges, sorted in decreasing order.
fig, ax = plt.subplots(figsize=(9, 5))
plot_ii_by_edges(
    pair_table,
    ax=ax,
    xlabel="Pozycja w rankingu\n(%)",
    ylabel="Informacja interakcyjna",
)
fig.tight_layout()
save_figure(fig, "medical_eda_interaction_information_by_edges")
plt.show()

## Interaction-information graph

The graph contains the top-$k$ feature pairs ranked by interaction information. Edge width and color encode interaction strength; node size reflects the feature's participation in strong pairs.

In [ ]:
n_cols = len(ii.columns)
full_graph_cols = n_cols * (n_cols - 1) / 2
n_cols, full_graph_cols

In [ ]:
# Graph of the k feature pairs with the highest II in the full dataset (from my_project.graphs).
top_pct = 0.2
k = int(full_graph_cols * top_pct)
fig, ax = plt.subplots(figsize=(13, 11))
# graph = plot_ii_graph(pair_table, ii.columns, k=k, ax=ax)

approx = False
if float(k) != (full_graph_cols * top_pct):
    print(f"Top {top_pct*100:.1f}% of edges is not an integer: {k} edges selected.")
    approx = True

selected_graph = plot_ii_graph(
    pair_table,
    ii.columns,
    k=k,
    ax=ax,
    # vmin=ii_vmin,
    # vmax=ii_vmax,
    layout="circular",
    curved_edges=False,
    label_fontsize=12,
    node_size=400,
    legend_label="II",
)

fig.tight_layout()
ax.set_title(
    (
        f"top {k} ({top_pct * 100:.0f}%) krawędzi"
        if not approx
        else f"top {k} (około {top_pct * 100:.0f}%) krawędzi"
    ),
    y=0.97,
    fontsize=14,
)
save_figure(fig, "medical_eda_interaction_graph")
plt.show()